# 05 · So sánh các VLM fine-tune

Giai đoạn cuối: gộp toàn bộ `eval_report.json` do notebook 03 sinh ra thành **một bảng
duy nhất**, rồi kiểm định thống kê trước khi kết luận model nào thắng.

**Điều kiện tiên quyết** — chạy xong notebook 03 cho cả 3 model (`qwen`, `internvl`,
`llama_vision`), mỗi model sinh 2 report (fine-tuned + zero-shot). Trong `result/` phải có:

```
eval_qwen_ft_back.json            eval_qwen_ft_back_preds.jsonl
eval_qwen_zeroshot_back.json      eval_qwen_zeroshot_back_preds.jsonl
eval_internvl_ft_back.json        ...
eval_llama_vision_ft_back.json    ...
```

Notebook này **không cần GPU** — chỉ đọc file JSON. Chọn runtime CPU cho nhanh.

> Vì sao phải kiểm định: tập test chỉ ~10% dữ liệu (200 ảnh). Chênh lệch vài phần trăm
> FA hoàn toàn có thể là nhiễu. Công bố "model X tốt hơn Y" mà không có khoảng tin cậy
> là kết luận không có cơ sở.

## 1. Mount Drive + kiểm kê report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import sys
from pathlib import Path

SIDE = 'Back'          # phải khớp notebook 03

DATA_DRIVE = Path('/content/drive/MyDrive/cccd_project/Data')
REPO_DRIVE = DATA_DRIVE / 'label_CCCD'

# [DA SUA] Doc report cua DUNG MOT mat the. compare_models.py gom nhom theo
# model_key x mode va KHONG phan biet Front/Back, nen de chung mot thu muc thi
# report cua mat nay bi coi la trung voi mat kia va bi bo qua.
LEGACY_DIR = DATA_DRIVE / 'result'
RESULT_DIR = LEGACY_DIR / SIDE.lower()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Tuong thich nguoc: notebook 03 ban cu ghi thang vao result/ (phang). Copy cac
# file cua dung mat dang xet sang result/{side}/. Dung copy chu khong move vi
# FUSE cua Drive xu ly rename khong tin cay; ban goc de lai vo hai.
copied = 0
for pattern in (f'eval_*_{SIDE.lower()}.json', f'eval_*_{SIDE.lower()}_preds.jsonl'):
    for p in LEGACY_DIR.glob(pattern):
        target = RESULT_DIR / p.name
        if not target.exists():
            shutil.copy2(p, target)
            copied += 1
if copied:
    print(f'Da copy {copied} file tu result/ -> result/{SIDE.lower()}/ (ban goc giu nguyen)')

sys.path.insert(0, str(REPO_DRIVE))
%cd {REPO_DRIVE}

print('Thu muc report:', RESULT_DIR)
print()
print('Report tim thay:')
found = sorted(RESULT_DIR.glob('eval_*.json'))
for p in found:
    print('  ', p.name)
if not found:
    print('   TRONG - chay notebook 03 truoc.')

print()
print('File du doan (can cho phan thong ke):')
preds = sorted(RESULT_DIR.glob('eval_*_preds.jsonl'))
for p in preds:
    print('  ', p.name)
if not preds:
    print('   TRONG - evaluate.py ban cu khong ghi preds. Chay lai notebook 03.')

## 2. Gộp thành bảng so sánh

`scripts/compare_models.py` tự gom nhóm theo `run.model_key` và tách theo `run.mode`
(zero_shot / fine_tuned), nên chỉ cần trỏ vào thư mục chứa report.

In [ ]:
!python scripts/compare_models.py --report_dir '{RESULT_DIR}' --report_path '{RESULT_DIR}/model_comparison.json' --markdown_path '{RESULT_DIR}/model_comparison.md'

In [ ]:
from IPython.display import Markdown, display

md_path = RESULT_DIR / 'model_comparison.md'
display(Markdown(md_path.read_text(encoding='utf-8')))

## 3. Khoảng tin cậy bootstrap cho Field Accuracy

**Resample theo ẢNH, không theo trường.** Các trường trong cùng một ảnh tương quan
mạnh với nhau (ảnh mờ thì sai cả loạt), nên coi mỗi trường là một quan sát độc lập sẽ
cho khoảng tin cậy hẹp giả tạo.

Cách đọc: nếu khoảng tin cậy của hai model **chồng lấn nhau**, không được kết luận
model nào tốt hơn — phải viết là "không có khác biệt có ý nghĩa thống kê".

In [ ]:
import json, math, random
from pathlib import Path
from src.utils.metrics import normalize_text

N_BOOT = 2000
SEED   = 42


def load_preds(path):
    """Doc file *_preds.jsonl -> list {image, pred, gold}."""
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def score_rows(rows):
    """
    Tra ve:
      per_image: [(so truong dung, tong so truong)] - don vi de bootstrap
      flat     : {(image, field): dung/sai}         - don vi de McNemar

    Dung DUNG normalize_text cua metrics.py de khop voi FA trong report.
    """
    per_image, flat = [], {}
    for r in rows:
        gold, pred = r.get('gold') or {}, r.get('pred') or {}
        n_ok = 0
        for k, gv in gold.items():
            ok = normalize_text(pred.get(k)) == normalize_text(gv)
            flat[(r['image'], k)] = ok
            n_ok += ok
        per_image.append((n_ok, len(gold)))
    return per_image, flat


def bootstrap_fa(per_image, n_boot=N_BOOT, seed=SEED):
    """CI 95% cho FA bang cluster bootstrap o muc anh."""
    rng, N, samples = random.Random(seed), len(per_image), []
    if N == 0:
        return (0.0, 0.0, 0.0)
    tot = sum(x[1] for x in per_image)
    point = sum(x[0] for x in per_image) / tot if tot else 0.0
    for _ in range(n_boot):
        draw = [per_image[rng.randrange(N)] for _ in range(N)]
        t = sum(x[1] for x in draw)
        samples.append(sum(x[0] for x in draw) / t if t else 0.0)
    samples.sort()
    return point, samples[int(0.025 * n_boot)], samples[int(0.975 * n_boot)]


# Nap moi file preds, gom theo (model_key, mode)
runs = {}
for p in sorted(RESULT_DIR.glob('eval_*_preds.jsonl')):
    report = p.with_name(p.name.replace('_preds.jsonl', '.json'))
    if not report.exists():
        print(f'Bo qua {p.name}: khong thay report tuong ung')
        continue
    run = json.loads(report.read_text(encoding='utf-8')).get('run', {})
    key = (run.get('model_key', p.stem), run.get('mode', 'fine_tuned'))
    per_image, flat = score_rows(load_preds(p))
    runs[key] = {'per_image': per_image, 'flat': flat, 'n_images': len(per_image)}

print(f'{"Model":<16}{"Mode":<12}{"N anh":>7}{"FA":>9}{"CI 95%":>22}')
print('-' * 68)
ci = {}
for key in sorted(runs):
    point, lo, hi = bootstrap_fa(runs[key]['per_image'])
    ci[key] = (point, lo, hi)
    print(f'{key[0]:<16}{key[1]:<12}{runs[key]["n_images"]:>7}'
          f'{point:>8.2%}   [{lo:.2%}, {hi:.2%}]')

## 4. McNemar test — so từng cặp model

Đây là phép kiểm định **ghép cặp**: hai model được chấm trên đúng cùng một tập ảnh, nên
chỉ cần đếm những chỗ chúng **bất đồng**.

- `b` = model A đúng, B sai
- `c` = model A sai, B đúng
- Nếu hai model ngang nhau thì `b` và `c` phải xấp xỉ nhau ⇒ kiểm định nhị thức chính
  xác với p = 0.5 trên `b` trong tổng `b + c`

Mạnh hơn nhiều so với so hai tỉ lệ độc lập, vì nó loại bỏ phương sai do độ khó của
từng ảnh.

⚠ Chạy ở mức **từng trường**, mà các trường trong cùng một ảnh không hoàn toàn độc lập
⇒ p-value có xu hướng lạc quan hơn thực tế. Đọc kèm khoảng tin cậy ở mục 3.

In [ ]:
def mcnemar_exact(flat_a, flat_b):
    """
    McNemar exact (binomial) tren cac cap (anh, truong) ma ca 2 model deu co.

    Returns: (b, c, p_value)
      b = A dung & B sai, c = A sai & B dung
    """
    keys = set(flat_a) & set(flat_b)
    b = sum(1 for k in keys if flat_a[k] and not flat_b[k])
    c = sum(1 for k in keys if not flat_a[k] and flat_b[k])
    n = b + c
    if n == 0:
        return b, c, 1.0
    m = min(b, c)
    p = 2 * sum(math.comb(n, k) for k in range(m + 1)) / (2 ** n)
    return b, c, min(p, 1.0)


ft_keys = sorted(k for k in runs if k[1] == 'fine_tuned')
print('So tung cap model (deu o che do fine-tuned)')
print()
print(f'{"A vs B":<36}{"A hon":>7}{"B hon":>7}{"p-value":>11}  Ket luan')
print('-' * 88)
for i in range(len(ft_keys)):
    for j in range(i + 1, len(ft_keys)):
        a, bk = ft_keys[i], ft_keys[j]
        b, c, p = mcnemar_exact(runs[a]['flat'], runs[bk]['flat'])
        verdict = ('khac biet CO y nghia (p<0.05)' if p < 0.05
                   else 'KHONG du bang chung de ket luan')
        print(f'{a[0] + " vs " + bk[0]:<36}{b:>7}{c:>7}{p:>11.4f}  {verdict}')

print()
print()
print('Fine-tuned vs Zero-shot cua CUNG mot model (dong gop cua QLoRA)')
print()
print(f'{"Model":<20}{"FT hon":>8}{"ZS hon":>8}{"p-value":>11}  Ket luan')
print('-' * 78)
for mk in sorted({k[0] for k in runs}):
    if (mk, 'fine_tuned') in runs and (mk, 'zero_shot') in runs:
        b, c, p = mcnemar_exact(runs[(mk, 'fine_tuned')]['flat'],
                                runs[(mk, 'zero_shot')]['flat'])
        verdict = 'QLoRA co tac dung ro ret' if p < 0.05 else 'chua du bang chung'
        print(f'{mk:<20}{b:>8}{c:>8}{p:>11.4f}  {verdict}')

## 5. Bảng cuối cho báo cáo

Gộp FA + khoảng tin cậy + các chỉ số vận hành thành một bảng markdown, lưu vào Drive
để dán thẳng vào báo cáo.

In [ ]:
summary = json.loads((RESULT_DIR / 'model_comparison.json').read_text(encoding='utf-8'))

lines = [
    f'# Bang ket qua so sanh VLM - CCCD mat {SIDE}',
    '',
    'Moi dong dung cung tap test, cung prompt/schema, cung `metrics.evaluate`, '
    f'cung greedy decoding. CI 95% bootstrap resample theo anh, {N_BOOT} lan, seed={SEED}.',
    '',
    '| Model | FA zero-shot | FA fine-tuned (CI 95%) | Delta FA | CER | micro-F1 | '
    'JSON parse | VRAM (GB) | p50/p95 (ms) |',
    '|' + '---|' * 9,
]

for mk in sorted(summary):
    modes = summary[mk]
    ft, zs = modes.get('fine_tuned'), modes.get('zero_shot')
    if not ft:
        continue
    point, lo, hi = ci.get((mk, 'fine_tuned'), (ft['field_accuracy'], None, None))
    ci_txt = f'{point:.2%} [{lo:.2%}, {hi:.2%}]' if lo is not None else f'{point:.2%}'
    fa_zs = f'{zs["field_accuracy"]:.2%}' if zs else '-'
    delta = f'{modes["delta_fa"]:+.2%}' if 'delta_fa' in modes else '-'
    lat = ft.get('latency_ms') or {}
    vram = f'{ft["peak_vram_mb"] / 1024:.1f}' if ft.get('peak_vram_mb') else '-'
    lines.append(
        f'| {ft["run"].get("model_id", mk)} | {fa_zs} | {ci_txt} | {delta} | '
        f'{ft["cer"]:.4f} | {ft["f1"]:.2%} | {ft.get("json_parse_rate", 0):.1%} | '
        f'{vram} | {lat.get("p50", "-")} / {lat.get("p95", "-")} |'
    )

lines += [
    '',
    '> Delta FA = FA(fine-tuned) - FA(zero-shot): dong gop thuan cua QLoRA.',
    '> Khoang tin cay chong lan => khong ket luan model nao tot hon.',
    '',
]

out = RESULT_DIR / f'final_comparison_{SIDE.lower()}.md'
out.write_text(chr(10).join(lines), encoding='utf-8')
print('Da luu ->', out)
print()
display(Markdown(chr(10).join(lines)))

## 6. Checklist trước khi viết kết luận

- [ ] Đủ 6 run (3 model × {zero-shot, fine-tuned})? Bảng ở mục 2 có phần "Còn thiếu".
- [ ] `trainable_params_pct` của 3 model có cùng cỡ không? Lệch nhiều nghĩa là ngân
      sách LoRA không tương đương — phải giải thích trong báo cáo.
- [ ] `image_tokens` từ cell probe của notebook 02 có cùng cỡ không? Đây là biến kiểm
      soát về lượng thông tin thị giác mỗi model nhận được.
- [ ] Khoảng tin cậy có chồng lấn không? Chồng lấn ⇒ viết "không có khác biệt có ý
      nghĩa thống kê", KHÔNG viết "model X tốt hơn".
- [ ] Số epoch có giống nhau giữa 3 model không?
- [ ] Đã chạy ≥ 2 seed chưa? Nếu chưa, ghi rõ trong phần hạn chế của báo cáo.